# Figure S2 Stage-wise violin plots

Nine representative genes from the accepted stage heatmap. Original RNA expression and original neuronal annotations are preserved. Full rebuilding and cached redraw both reproduce the selected figure.

Layout reference: [STAGATE Fig. 6c](https://www.nature.com/articles/s41467-022-29439-6/figures/6), with [Nature Medicine Extended Data Fig. 3a-b](https://www.nature.com/articles/s41591-024-03215-z) as another stacked-violin example. Colors use the accepted heatmap's frozen six-node palette and Z scale; the papers do not establish this atlas's temporal specificity.

## 1. Shared configuration

Run from the project root. Outputs use the same directories as the heatmap. PNG is rendered from the final vector PDF at 600 DPI. Dependencies: Anndata, NumPy, Pandas, Matplotlib, SciPy, Pillow, pypdf, pywin32, Poppler, and Illustrator.

In [ ]:
#1. =========== Paths and shared settings
from pathlib import Path
import os

# Replace each ROOT placeholder with the required file or directory path before running.
ROOT = Path(".")
if ROOT.name == "Code_def":
    ROOT = ROOT.parent
# Uses CSS_3000_Embryonic_Inte_adata.h5ad generated by fig1_Embryonic_integration.ipynb.
RNA_FILE = ROOT
# Uses Stage-heatmap.h5ad generated by fig2_embry_heatmap.ipynb.
HEATMAP_FILE = ROOT
# Set the file or directory path.
TMP_DIR = ROOT
STAGE, STUDY, STATE = "stage_standardized", "study_id", "cell_type_plot"
SEED, PNG_DPI = 0, 600
# Set the file or directory path.
FIG_DIR = ROOT
REBUILD_RESULTS = False              # True rebuilds the violin cache from original RNA.
TMP_DIR.mkdir(parents=True, exist_ok=True)
# Set the file or directory path.
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1", MPLCONFIGDIR=str(ROOT))

import anndata as ad
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable



import json, hashlib, tempfile, subprocess, shutil
from importlib.metadata import version
from matplotlib.collections import QuadMesh
from matplotlib.text import Text
from matplotlib.colors import to_rgba
from scipy.interpolate import PchipInterpolator
from PIL import Image as PILImage
from pypdf import PdfReader, PdfWriter
from pypdf.generic import ContentStream, DecodedStreamObject, NameObject
import win32com.client.dynamic as com
from IPython.display import display, Image

# Set the executable or tool directory path.
POPPLER = ROOT

import re
from pypdf._codecs import charset_encoding
from pypdf._codecs.adobe_glyphs import adobe_glyphs
from pypdf.generic import FloatObject, IndirectObject, StreamObject
from pypdf.generic import DictionaryObject


## 2. Original RNA and weighted distributions

The accepted nine-gene panel is fixed. Rebuilding reads the integration RNA object and frozen heatmap scaling. BANDWIDTH=0.22 controls KDE smoothing in log1p(CP10k) units; GRID_SIZE=180 controls density sampling. Changing these or the gene panel requires REBUILD_RESULTS=True, followed by cells 2 onward. No DEG or GO is run.

In [ ]:
#2. =========== Rebuild the selected RNA distributions
GENES = ["Sox2", "Neurod1", "Insm1", "Shox2", "Shh", "Gata3", "Id1", "Plxna1", "Calb2"]
CANDIDATES = {"Id1"}                 # Auditory developmental role remains a candidate.
PLXNA1_REFERENCE = "https://pmc.ncbi.nlm.nih.gov/articles/PMC6697390/"


BANDWIDTH = 0.22                     # Absolute log1p(CP10k) bandwidth; shared across stages.
GRID_SIZE = 180                      # Density evaluation points within each gene's range.

def rna_density(values, cell_weights, grid):
    if np.ptp(values) < 1e-7:
        return np.zeros_like(grid)
    # Include measured zero values, reflect at zero, and cut at the observed range.
    distance = (grid[:, None] - values[None, :]) / BANDWIDTH
    reflected = (grid[:, None] + values[None, :]) / BANDWIDTH
    density = (np.exp(-0.5 * distance**2) + np.exp(-0.5 * reflected**2)) @ cell_weights
    density /= BANDWIDTH * np.sqrt(2 * np.pi) * cell_weights.sum()
    density[(grid < values.min()) | (grid > values.max())] = 0
    area = np.trapezoid(density, grid)
    return density / area if area > 0 else density


def build_violin_data(summary):
    STAGES = list(summary.uns["stage_order"])
    STAGE_COLORS = dict(summary.uns["stage_colors"])
    labels = summary.uns["marker_labels"].set_index("gene_symbol").loc[GENES].copy()
    labels["candidate"] = labels.index.isin(CANDIDATES)
    labels.loc["Plxna1", "reference"] = PLXNA1_REFERENCE
    labels.loc["Plxna1", "literature_evidence"] = "E17.5 SGN expression and SGN-specific conditional knockout"
    gene_ids = labels.gene_id.to_list()
    assert len(GENES) == len(STAGES) and labels.stage.to_list() == STAGES

    rna = ad.read_h5ad(RNA_FILE, backed="r")
    obs = rna.obs.loc[rna.obs.source_state_evaluation_allowed.astype(bool)].copy()
    obs["unit"] = obs.biological_replicate_id.astype(str).where(obs.biological_replicate_id.astype(str) != "Not available", obs.library_id.astype(str))
    for key in [STAGE, STUDY, "unit"]:
        obs[key] = obs[key].astype(str)
    matrix = rna[obs.index, gene_ids].X
    expression = matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)
    rna.file.close()
    assert len(obs) == 6995 and np.isfinite(expression).all() and expression.min() >= 0
    # X is already log1p(CP10k): do not renormalize or use CSS/PC/UMAP coordinates.



    # Each study has equal mass within a stage; each unit has equal mass within a study.
    # Cell weights preserve the original state composition within each unit.
    obs["n_unit_cells"] = obs.groupby([STAGE, STUDY, "unit"], observed=True)["unit"].transform("size")
    obs["n_units"] = obs.groupby([STAGE, STUDY], observed=True)["unit"].transform("nunique")
    obs["n_studies"] = obs.groupby(STAGE, observed=True)[STUDY].transform("nunique")
    weights = (1 / (obs.n_unit_cells * obs.n_units * obs.n_studies)).to_numpy()
    stage_indices = [np.flatnonzero(obs[STAGE].to_numpy() == stage) for stage in STAGES]
    means = np.array([np.average(expression[ids], axis=0, weights=weights[ids]) for ids in stage_indices])
    detection = np.array([np.average(expression[ids] > 0, axis=0, weights=weights[ids]) for ids in stage_indices])
    assert np.allclose([weights[ids].sum() for ids in stage_indices], 1)
    assert np.argmax(means, axis=0).tolist() == list(range(len(GENES)))
    cell_counts = [len(ids) for ids in stage_indices]
    unit_counts = [len(obs.iloc[ids][[STUDY, "unit"]].drop_duplicates()) for ids in stage_indices]



    grids, densities = [], []
    for j in range(len(GENES)):
        grid = np.linspace(0, expression[:, j].max(), GRID_SIZE)
        grids.append(grid)
        densities.append(np.array([rna_density(expression[ids, j], weights[ids], grid) for ids in stage_indices]))



    gene_positions = summary.var_names.get_indexer(gene_ids)
    selected_obs = obs[[STAGE, STUDY, STATE, "unit", "library_id", "biological_replicate_id", "technology"]].copy()
    selected_obs.index = pd.Index(selected_obs.index.to_numpy(dtype=str), dtype=object, name="cell_id")
    gene_table = labels.reset_index()
    for frame in [selected_obs, gene_table]:
        for column in frame.select_dtypes(include=["string"]).columns:
            frame[column] = frame[column].astype(object)
    selected_obs["weight"] = weights
    selected_obs["unit_kind"] = np.where(selected_obs[STUDY] == "GSE195500", "biological_replicate", "library_or_plate")
    for column in selected_obs.columns.difference(["weight"]):
        selected_obs[column] = pd.Categorical(selected_obs[column])
    return {
        "expression": expression, "obs": selected_obs, "genes": gene_table,
        "means": means, "detection_fraction": detection,
        "grids": np.asarray(grids), "densities": np.asarray(densities),
        "row_mean": np.asarray(summary.uns["row_mean_log1p_CP10k"])[gene_positions],
        "row_sd": np.asarray(summary.uns["row_sd_log1p_CP10k"])[gene_positions],
        "stage_order": np.asarray(STAGES, dtype=object), "stage_colors": STAGE_COLORS,
        "plot_settings": summary.uns["plot_settings"],
        "parameters": {"bandwidth": BANDWIDTH, "grid_size": GRID_SIZE, "width_normalization": "per_violin_max", "seed": SEED},
        "input_file": str(RNA_FILE.relative_to(ROOT)), "input_sha256": summary.uns["input_sha256"],
        "methods": {
            "expression": "Original X=log1p(CP10k); measured zeros retained; no additional normalization.",
            "weights": "Studies equal within stage; units equal within study; cells equal within unit; original-state proportions retained.",
            "density": "Gaussian KDE includes zero values; reflection at zero; observed-range cut; per-violin maximum width.",
            "color": "Stage-balanced mean original RNA standardized by frozen heatmap row mean/SD; fixed +/-3.5 and original six color nodes.",
            "evidence": "Selected qualified heatmap genes; no new differential tests. E9.5/E17.5 each have one library; stage/source/composition effects remain mixed."
        },
        "references": {
            "layout": "https://www.nature.com/articles/s41467-022-29439-6/figures/6",
            "stacked_violin": "https://www.nature.com/articles/s41591-024-03215-z",
            "Plxna1": PLXNA1_REFERENCE
        },
        "software": {"anndata": version("anndata"), "numpy": np.__version__, "pandas": pd.__version__, "matplotlib": matplotlib.__version__}
    }


## 3. Essential cache

Only uns['stage_violin'] is added to the existing heatmap H5AD. Original heatmap arrays, tables and metadata are checked before atomic replacement; the DEG table checksum stays identical. This keeps the result directory at two files. Default cache loading avoids recomputing RNA distributions.

In [ ]:

#3. =========== Store or reuse the essential violin data
def assert_same(a, b):
    if isinstance(a, dict):
        assert set(a) == set(b)
        for key in a: assert_same(a[key], b[key])
    elif isinstance(a, pd.DataFrame):
        pd.testing.assert_frame_equal(a, b)
    elif isinstance(a, np.ndarray):
        assert a.dtype == b.dtype
        np.testing.assert_equal(a, b)
    else:
        assert a == b

summary = ad.read_h5ad(HEATMAP_FILE)
if REBUILD_RESULTS or "stage_violin" not in summary.uns:
    # Uses Stage-DEG.tsv.gz generated by fig2_embry_heatmap.ipynb.
    DEG_FILE = ROOT
    deg_digest = hashlib.sha256(DEG_FILE.read_bytes()).hexdigest()
    data = build_violin_data(summary)
    updated = summary.copy()
    updated.uns["stage_violin"] = data
    with tempfile.TemporaryDirectory(prefix="cache-", dir=TMP_DIR) as td:
        # Uses Stage-heatmap.h5ad generated by fig2_embry_heatmap.ipynb.
        candidate = ROOT
        updated.write_h5ad(candidate, compression="gzip", convert_strings_to_categoricals=False)
        verified = ad.read_h5ad(candidate)
        assert_same(summary.X, verified.X)
        assert_same(dict(summary.layers), dict(verified.layers))
        pd.testing.assert_frame_equal(summary.obs, verified.obs)
        pd.testing.assert_frame_equal(summary.var, verified.var)
        for key in summary.uns:
            if key != "stage_violin": assert_same(summary.uns[key], verified.uns[key])
        assert_same(data, verified.uns["stage_violin"])
        assert hashlib.sha256(DEG_FILE.read_bytes()).hexdigest() == deg_digest
        candidate.replace(HEATMAP_FILE)
    data = verified.uns["stage_violin"]
else:
    data = summary.uns["stage_violin"]
assert data["genes"].gene_symbol.to_list() == GENES
assert data["parameters"]["bandwidth"] == BANDWIDTH and data["parameters"]["grid_size"] == GRID_SIZE
print("Violin cache:", data["expression"].shape, "| stored in Stage-heatmap.h5ad.uns['stage_violin'].")


## 4. Appearance

Change figure size, density width or fonts here and rerun cells 4 onward. Height is original RNA; each gene shares one range across stages. Every violin uses the same maximum width; width describes density. Color is a stage-balanced mean on the frozen heatmap Z scale. All stages omit raw-cell scatter; hollow circles show the study- and unit-balanced means. Figure labels and literature links distinguish known evidence from Id1 as a candidate.

In [ ]:
#4. =========== Final violin appearance
FIGSIZE = (6.4, 6.8)                 # Inches; compact nine-gene panel.
MAX_HALF_WIDTH = 0.40                # Standard width normalization for each violin.
FONT_SIZE = 8


def make_violin_layout(data):
    GENES = data["genes"].gene_symbol.to_list()
    CANDIDATES = set(data["genes"].loc[data["genes"].candidate, "gene_symbol"])
    STAGES, STAGE_COLORS = list(data["stage_order"]), dict(data["stage_colors"])
    obs, expression, means = data["obs"], data["expression"], data["means"]
    grids, densities = data["grids"], data["densities"]
    stage_indices = [np.flatnonzero(obs[STAGE].to_numpy() == stage) for stage in STAGES]
    cell_counts = [len(ids) for ids in stage_indices]
    unit_counts = [len(obs.iloc[ids][[STUDY, "unit"]].drop_duplicates()) for ids in stage_indices]
    # Stacked-violin layout reference: STAGATE, Nature Communications (2022), Fig. 6c.
    # Use the same centering, scale, colors and limits as the heatmap.
    settings = data["plot_settings"]
    Z_LIMIT = float(settings["z_limit"])
    nodes = (np.asarray(settings["color_values"]) + Z_LIMIT) / (2 * Z_LIMIT)
    palette = LinearSegmentedColormap.from_list("embry_heatmap", list(zip(nodes, settings["colors"])), N=256)
    color_norm = Normalize(-Z_LIMIT, Z_LIMIT, clip=True)
    row_mean, row_sd = data["row_mean"], data["row_sd"]
    color_z = (means - row_mean[None, :]) / row_sd[None, :]
    violin_colors = palette(color_norm(color_z))
    # Color is the stage-balanced mean RNA on the frozen heatmap scale, not cell density.

    plt.rcParams.update({"font.family": "Arial", "font.size": FONT_SIZE, "axes.linewidth": 0.5, "pdf.fonttype": 42, "ps.fonttype": 42})
    fig, axes = plt.subplots(len(GENES), 1, figsize=FIGSIZE, sharex=True, gridspec_kw={"hspace": 0.19})
    fig.subplots_adjust(left=0.15, right=0.94, bottom=0.16, top=0.81)
    for j, (gene, ax) in enumerate(zip(GENES, axes)):
        grid, density_set = grids[j], densities[j]
        for k, ids in enumerate(stage_indices):
            values, density = expression[ids, j], density_set[k]
            fill_color = violin_colors[k, j]
            edge_color = np.r_[fill_color[:3] * 0.80, 1.0]
            ax.plot([k - MAX_HALF_WIDTH, k + MAX_HALF_WIDTH], [0, 0], color="0.86", lw=0.5, zorder=1, clip_on=False)
            if density.max() > 0:
                observed = (grid >= values.min()) & (grid <= values.max())
                visible_grid = grid[observed]
                widths = MAX_HALF_WIDTH * density[observed] / density.max()
                vertices = np.column_stack([np.r_[k - widths, (k + widths)[::-1]], np.r_[visible_grid, visible_grid[::-1]]])
                ax.fill(vertices[:, 0], vertices[:, 1], facecolor=fill_color, edgecolor=edge_color, alpha=1.0, lw=0.45, zorder=2)
            else:
                ax.plot([k - 0.12, k + 0.12], [values[0], values[0]], color=edge_color, lw=1.1, zorder=2, clip_on=False)
            ax.plot(k, means[k, j], marker="o", ms=2.9, mfc="white", mec="0.18", mew=0.55, linestyle="none", zorder=4, clip_on=False)
        ax.set_ylim(-0.04 * grid.max(), 1.06 * grid.max())
        ax.set_yticks([0, grid.max()], labels=["0", f"{grid.max():.1f}"])
        ax.yaxis.tick_right()
        ax.tick_params(axis="y", length=2, pad=2, labelsize=6.8)
        ax.tick_params(axis="x", length=0, labelbottom=False)
        ax.set_ylabel(gene + ("*" if gene in CANDIDATES else ""), rotation=0, fontstyle="italic", ha="right", va="center", fontsize=9, labelpad=13)
        ax.yaxis.set_label_coords(-0.03, 0.43)
        for side in ["left", "right", "top", "bottom"]:
            ax.spines[side].set_visible(False)
        ax.set_xlim(-0.55, len(STAGES) - 0.45)

    axes[-1].set_xticks(range(len(STAGES)), labels=STAGES)
    axes[-1].tick_params(axis="x", labelbottom=True, labelsize=8, pad=7)
    axes[-1].set_xlabel("Real embryonic stage", fontsize=9, labelpad=8)
    fig.suptitle("Stage-associated RNA expression", x=0.15, y=0.962, ha="left", fontsize=12)
    fig.text(0.15, 0.929, "CVG / SGN atlas  |  original neuronal states retained", fontsize=8.2, color="0.30")
    color_ax = fig.add_axes([0.745, 0.949, 0.195, 0.012])
    colorbar = fig.colorbar(ScalarMappable(norm=color_norm, cmap=palette), cax=color_ax, orientation="horizontal", ticks=[-Z_LIMIT, 0, Z_LIMIT])
    color_ax.set_title("Stage mean Z\n(heatmap scaling)", fontsize=6.5, pad=3)
    color_ax.tick_params(labelsize=6.2, length=2, width=0.4, pad=1)
    colorbar.outline.set_linewidth(0.4)
    header_y = [0.889, 0.864, 0.839]
    for text, y in zip(["Stage", "Cells", "Units"], header_y):
        fig.text(0.112, y, text, fontsize=7.4, ha="right", va="center", color="0.35")
    for k, stage in enumerate(STAGES):
        x = fig.transFigure.inverted().transform(axes[0].transData.transform((k, 0)))[0]
        fig.text(x, header_y[0], stage, fontsize=8, color="0.22", ha="center", va="center", fontweight="bold")
        fig.add_artist(Line2D([x - 0.020, x + 0.020], [header_y[0] + 0.018] * 2, transform=fig.transFigure, color=STAGE_COLORS[stage], lw=3, solid_capstyle="butt"))
        fig.text(x, header_y[1], str(cell_counts[k]), fontsize=7.3, color="0.35", ha="center", va="center")
        fig.text(x, header_y[2], str(unit_counts[k]), fontsize=7.3, color="0.35", ha="center", va="center")
    mean_marker = Line2D([], [], marker="o", ms=3.2, mfc="white", mec="0.18", mew=0.6, linestyle="none")
    fig.legend([mean_marker], ["Study- and unit-balanced mean"], loc="lower left", bbox_to_anchor=(0.136, 0.075), frameon=False, fontsize=7.6, handletextpad=0.4)
    fig.text(0.15, 0.066, "Height: log1p(CP10k), same range per gene. Width: density; each violin has equal maximum width.", fontsize=6.7, color="0.35")
    fig.text(0.15, 0.046, "Units = biological replicates or libraries/plates. E9.5 / E17.5: one library; all zeros retained.", fontsize=6.7, color="0.35")
    fig.text(0.15, 0.026, "* Id1: auditory developmental candidate. Peaks describe this atlas; stage/source effects remain mixed.", fontsize=6.7, color="0.35")

    return fig


## 5. Editable vector export

Native Illustrator creates four sibling layers and page masks: Graphics, Legend, Text and Axes. Violins use cubic Bezier paths, markers are independent circles, guides are straight paths and labels remain complete Arial strings. The PDF's public clipping and Unicode maps are checked without changing Illustrator's private editable objects.

In [ ]:
#5. =========== Native editable vector export
def vector_scene(fig):
    fig.canvas.draw()
    renderer, scale = fig.canvas.get_renderer(), 72 / fig.dpi
    scene = {"size": [float(v * 72) for v in fig.get_size_inches()], "paths": [], "circles": [], "texts": []}

    def rgb(color):
        return [round(float(v) * 255, 4) for v in to_rgba(color)[:3]]

    def points(vertices, transform):
        return np.round(transform.transform(vertices) * scale, 4).tolist()

    def path(vertices, transform, layer, fill=None, stroke=None, width=0, closed=False, handles=None):
        p = points(vertices, transform)
        if closed and p[0] == p[-1]:
            p = p[:-1]
        scene["paths"].append({"p": p, "layer": layer, "fill": fill, "stroke": stroke, "width": float(width), "closed": closed,
                               "h": [points(h, transform) for h in handles] if handles is not None else None})

    def violin(artist):
        vertices = artist.get_xy()
        if np.array_equal(vertices[0], vertices[-1]):
            vertices = vertices[:-1]
        n = len(vertices) // 2
        if n < 2:
            path(vertices, artist.get_transform(), "Graphics", fill=rgb(artist.get_facecolor()),
                 stroke=rgb(artist.get_edgecolor()), width=artist.get_linewidth(), closed=True)
            return
        left, right = vertices[:n], vertices[n:]
        handles = []
        for side in [left, right]:
            increasing = side if side[0, 1] < side[-1, 1] else side[::-1]
            derivative = PchipInterpolator(increasing[:, 1], increasing[:, 0]).derivative()(side[:, 1])
            for i, anchor in enumerate(side):
                before, after = anchor.copy(), anchor.copy()
                if i > 0:
                    step = (side[i, 1] - side[i - 1, 1]) / 3
                    before -= [derivative[i] * step, step]
                if i < n - 1:
                    step = (side[i + 1, 1] - side[i, 1]) / 3
                    after += [derivative[i] * step, step]
                handles.append([before, after])
        path(vertices, artist.get_transform(), "Graphics", fill=rgb(artist.get_facecolor()),
             stroke=rgb(artist.get_edgecolor()), width=artist.get_linewidth(), closed=True, handles=handles)

    def circle(point, line, layer):
        face, edge = line.get_markerfacecolor(), line.get_markeredgecolor()
        scene["circles"].append({"x": round(float(point[0]) * scale, 4), "y": round(float(point[1]) * scale, 4),
                                "r": float(line.get_markersize()) / 2, "fill": None if face == "none" else rgb(face),
                                "stroke": None if edge == "none" else rgb(edge), "width": float(line.get_markeredgewidth()),
                                "opacity": 100 * (line.get_alpha() if line.get_alpha() is not None else 1), "layer": layer})

    def line(artist, layer):
        if not artist.get_visible():
            return
        if artist.get_linestyle() not in ["None", "", " "]:
            path(artist.get_path().vertices, artist.get_transform(), layer, stroke=rgb(artist.get_color()), width=artist.get_linewidth())
        if artist.get_marker() == "o":
            vertices = artist.get_transform().transform(artist.get_xydata())
            selected = artist.get_markevery()
            if isinstance(selected, list):
                vertices = vertices[selected]
            for point in vertices:
                circle(point, artist, layer)

    def text(artist, layer):
        if not artist.get_visible() or not artist.get_text():
            return
        _, lines, _ = artist._get_layout(renderer)
        x, y = artist.get_transform().transform(artist.get_unitless_position())
        font = artist.get_fontproperties()
        face = "Arial-ItalicMT" if font.get_style() == "italic" else ("Arial-BoldMT" if font.get_weight() in ["bold", 700] else "ArialMT")
        for content, _, ox, oy in lines:
            scene["texts"].append({"text": content, "x": round(float(x + ox) * scale, 4), "y": round(float(y + oy) * scale, 4),
                                   "size": float(font.get_size_in_points()), "font": face, "color": rgb(artist.get_color()), "layer": layer})

    def ticks(axis, layer):
        for tick in axis.get_major_ticks():
            text(tick.label1, layer)
            text(tick.label2, layer)
            for item in [tick.tick1line, tick.tick2line]:
                if not item.get_visible() or item.get_markersize() <= 0:
                    continue
                direction = {0: [-1, 0], 1: [1, 0], 2: [0, 1], 3: [0, -1]}[item.get_marker()]
                point = item.get_transform().transform(item.get_path().vertices[0])
                path([point, point + np.asarray(direction) * item.get_markersize() / scale], matplotlib.transforms.IdentityTransform(),
                     "Legend" if layer == "Legend" else "Axes", stroke=rgb(item.get_color()), width=item.get_markeredgewidth())

    for i, ax in enumerate(fig.axes):
        layer = "Legend" if i == len(fig.axes) - 1 else "Graphics"
        for mesh in ax.collections:
            if isinstance(mesh, QuadMesh):
                for shape, color in zip(mesh.get_paths(), mesh.get_facecolors()):
                    path(shape.vertices, mesh.get_transform(), layer, fill=rgb(color), closed=True)
        for artist in ax.patches:
            violin(artist)
        for item in ax.lines:
            line(item, "Axes" if isinstance(item.get_color(), str) and item.get_color() == "0.86" else layer)
        for spine in ax.spines.values():
            if spine.get_visible():
                path(spine.get_path().vertices, spine.get_transform(), layer, stroke=rgb(spine.get_edgecolor()), width=spine.get_linewidth(), closed=True)
        for artist in ax.texts:
            text(artist, "Text")
        text(ax.title, "Legend" if layer == "Legend" else "Text")
        text(ax.xaxis.label, "Text")
        text(ax.yaxis.label, "Text")
        ticks(ax.xaxis, "Legend" if layer == "Legend" else "Text")
        ticks(ax.yaxis, "Legend" if layer == "Legend" else "Text")
    for artist in fig.artists:
        if isinstance(artist, Line2D):
            line(artist, "Graphics")
    for legend in fig.legends:
        for item in legend.legend_handles:
            line(item, "Legend")
        for artist in legend.get_texts():
            text(artist, "Legend")
    for artist in fig.texts:
        text(artist, "Text")
    return scene


def verify_illustrator(app, pdf, scene):
    prefix = "var S=" + json.dumps({"pdf": str(pdf)}, ensure_ascii=True) + ";\n"
    helper = ILLUSTRATOR_JS[:ILLUSTRATOR_JS.index("    var doc = app.documents.add")]
    verification = ILLUSTRATOR_JS[ILLUSTRATOR_JS.index("        var check=app.open"):ILLUSTRATOR_JS.index("    } finally")]
    report = json.loads(app.DoJavaScript(prefix + helper + "\ntry {\n" + verification + "\n} finally {app.userInteractionLevel=oldLevel;}\n})();"))
    assert [x["name"] for x in report["layers"]] == ["Text", "Legend", "Graphics", "Axes"]
    assert all(x["groups"] == x["masks"] == 1 for x in report["layers"])
    assert report["clipping_groups"] == report["clipping_paths"] == 4
    assert report["raster_items"] == report["placed_items"] == 0
    assert sorted(report["texts"]) == sorted(x["text"] for x in scene["texts"])
    assert set(report["fonts"]) <= {"ArialMT", "Arial-BoldMT", "Arial-ItalicMT"}
    return report


def export_violin(fig):
    scene = vector_scene(fig)
    # Set the file or directory path.
    pdf, png = ROOT, ROOT
    scene.update(pdf=str(pdf))
    app = com.Dispatch("Illustrator.Application.29")
    with tempfile.TemporaryDirectory(prefix="illustrator-", dir=TMP_DIR) as td:
        td = Path(td)
        # Set the file or directory path.
        script = ROOT
        script.write_text("var S=" + json.dumps(scene, ensure_ascii=True, separators=(",", ":")) + ";\n" + ILLUSTRATOR_JS, encoding="utf-8")
        app.DoJavaScriptFile(str(script))
        # Set the file or directory path.
        repaired = ROOT
        repair_native_pdf(pdf, repaired)
        repaired.replace(pdf)
        report = verify_illustrator(app, pdf, scene)
    with tempfile.TemporaryDirectory(prefix="png-", dir=TMP_DIR) as td:
        # Set the file or directory path.
        rendered = ROOT
        subprocess.run([POPPLER, "-r", str(PNG_DPI), "-png", "-singlefile", str(pdf), str(rendered)], check=True, capture_output=True)
        with PILImage.open(rendered.with_suffix(".png")) as image:
            image.load()
            expected = tuple(round(v * PNG_DPI / 72) for v in scene["size"])
            assert image.size == expected
            image.save(png, dpi=(PNG_DPI, PNG_DPI))
    plt.close(fig)
    print("Verified Illustrator:", len(report["texts"]), "complete Arial texts, 4 layers, 4 masks, no raster objects.")
    print("PNG:", expected, "at", PNG_DPI, "DPI.")
    return report


ILLUSTRATOR_JS = '\n(function () {\n    var oldLevel = app.userInteractionLevel;\n    app.userInteractionLevel = UserInteractionLevel.DONTDISPLAYALERTS;\n    function rgb(values) {var c=new RGBColor();c.red=values[0];c.green=values[1];c.blue=values[2];return c;}\n    function json(v) {\n        if (v === null) return "null";\n        if (typeof v === "number" || typeof v === "boolean") return String(v);\n        if (typeof v === "string") return \'"\'+v.replace(/\\\\/g,\'\\\\\\\\\').replace(/"/g,\'\\\\"\').replace(/\\r/g,\'\\\\r\').replace(/\\n/g,\'\\\\n\')+\'"\';\n        var parts=[];if(v instanceof Array){for(var i=0;i<v.length;i++)parts.push(json(v[i]));return \'[\'+parts.join(\',\')+\']\';}\n        for(var k in v)if(v.hasOwnProperty(k))parts.push(json(k)+\':\'+json(v[k]));return \'{\'+parts.join(\',\')+\'}\';\n    }\n    var doc = app.documents.add(DocumentColorSpace.RGB, S.size[0], S.size[1]);\n    try {\n        var groups={}, masks={}, names=["Axes", "Graphics", "Legend", "Text"];\n        for(var i=0;i<names.length;i++) {\n            var layer=i===0?doc.layers[0]:doc.layers.add();layer.name=names[i];\n            var group=layer.groupItems.add();group.name=names[i];groups[names[i]]=group;\n            var mask=group.pathItems.rectangle(S.size[1],0,S.size[0],S.size[1]);\n            mask.name="Page mask";mask.filled=false;mask.stroked=false;mask.clipping=true;masks[names[i]]=mask;\n        }\n        for(var i=0;i<S.paths.length;i++) {\n            var r=S.paths[i], p=groups[r.layer].pathItems.add();p.setEntirePath(r.p);p.closed=r.closed;\n            if(r.h){for(var j=0;j<r.h.length;j++){p.pathPoints[j].leftDirection=r.h[j][0];p.pathPoints[j].rightDirection=r.h[j][1];}}\n            p.filled=r.fill!==null;if(p.filled)p.fillColor=rgb(r.fill);\n            p.stroked=r.stroke!==null;if(p.stroked){p.strokeColor=rgb(r.stroke);p.strokeWidth=r.width;}\n        }\n        for(var i=0;i<S.circles.length;i++) {\n            var r=S.circles[i],p=groups[r.layer].pathItems.ellipse(r.y+r.r,r.x-r.r,r.r*2,r.r*2);\n            p.filled=r.fill!==null;if(p.filled)p.fillColor=rgb(r.fill);\n            p.stroked=r.stroke!==null;if(p.stroked){p.strokeColor=rgb(r.stroke);p.strokeWidth=r.width;}\n            p.opacity=r.opacity;\n        }\n        for(var i=0;i<S.texts.length;i++) {\n            var r=S.texts[i],t=groups[r.layer].textFrames.pointText([r.x,r.y]);t.contents=r.text;\n            t.textRange.characterAttributes.textFont=app.textFonts.getByName(r.font);\n            t.textRange.characterAttributes.size=r.size;t.textRange.characterAttributes.fillColor=rgb(r.color);\n        }\n        for(var i=0;i<names.length;i++){masks[names[i]].zOrder(ZOrderMethod.BRINGTOFRONT);groups[names[i]].clipped=true;}\n        var opt=new PDFSaveOptions();opt.compatibility=PDFCompatibility.ACROBAT6;\n        opt.preserveEditability=true;opt.acrobatLayers=true;opt.generateThumbnails=false;opt.viewAfterSaving=false;\n        doc.saveAs(new File(S.pdf),opt);\n        doc.close(SaveOptions.DONOTSAVECHANGES);doc=null;\n        var check=app.open(new File(S.pdf));\n        var report={version:app.version,layers:[],clipping_groups:0,clipping_paths:0,raster_items:check.rasterItems.length,placed_items:check.placedItems.length,texts:[],fonts:[],paths:check.pathItems.length};\n        for(var i=0;i<check.layers.length;i++){\n            var layer=check.layers[i],item={name:layer.name,groups:layer.groupItems.length,masks:0};\n            for(var j=0;j<layer.groupItems.length;j++)if(layer.groupItems[j].clipped){report.clipping_groups++;item.masks++;}\n            report.layers.push(item);\n        }\n        for(var i=0;i<check.pathItems.length;i++)if(check.pathItems[i].clipping)report.clipping_paths++;\n        var seen={};for(var i=0;i<check.textFrames.length;i++){\n            var t=check.textFrames[i];report.texts.push(t.contents);var f=t.textRange.characterAttributes.textFont.name;\n            if(!seen[f]){seen[f]=true;report.fonts.push(f);}\n        }\n        check.close(SaveOptions.DONOTSAVECHANGES);\n        return json(report);\n    } finally {if(doc)doc.close(SaveOptions.DONOTSAVECHANGES);app.userInteractionLevel=oldLevel;}\n})()\n'

def repair_native_pdf(source: Path, destination: Path):
    """Add public PDF clips and Unicode maps without altering Illustrator data."""
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if source == destination:
        raise ValueError("Repair must write a separate PDF before replacement.")
    reader = PdfReader(source, strict=True)
    expected = {"Graphics", "Axes", "Text", "Legend"}
    ocgs = reader.trailer["/Root"]["/OCProperties"]["/OCGs"]
    if len(ocgs) != 4 or {str(x.get_object()["/Name"]) for x in ocgs} != expected:
        raise ValueError("Native source must have the four expected OCGs.")

    def canonical(value, active=None):
        active = set() if active is None else active
        if isinstance(value, IndirectObject):
            value = value.get_object()
        if isinstance(value, dict):
            identity = id(value)
            if identity in active:
                raise ValueError("Unexpected cycle in preserved Illustrator metadata.")
            active = active | {identity}
            result = {str(k): canonical(v, active) for k, v in sorted(value.items())}
            if isinstance(value, StreamObject):
                result["stream_sha256"] = hashlib.sha256(value.get_data()).hexdigest()
                result["encoded_sha256"] = hashlib.sha256(value._data).hexdigest()
            return result
        if isinstance(value, (list, tuple)):
            return [canonical(v, active) for v in value]
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if isinstance(value, (int, float, bool)) or value is None:
            return value
        return str(value)

    def private_signature(pdf):
        root = pdf.trailer["/Root"]
        values = {
            "catalog": {str(k): canonical(v) for k, v in root.items() if k != "/Pages"},
            "page_pieceinfo": [canonical(p.get("/PieceInfo", {})) for p in pdf.pages],
            "info": canonical(pdf.trailer.get("/Info", {})),
            "ID": canonical(pdf.trailer.get("/ID", [])),
        }
        return hashlib.sha256(json.dumps(values, sort_keys=True).encode()).hexdigest()

    def multiply(a, b):
        a0, a1, a2, a3, a4, a5 = a
        b0, b1, b2, b3, b4, b5 = b
        return (
            a0*b0+a2*b1, a1*b0+a3*b1, a0*b2+a2*b3,
            a1*b2+a3*b3, a0*b4+a2*b5+a4, a1*b4+a3*b5+a5,
        )

    def inverse(a):
        x, y, z, w, u, v = a
        determinant = x*w-y*z
        if abs(determinant) < 1e-12:
            raise ValueError("Singular PDF transformation matrix.")
        return (
            w/determinant, -y/determinant, -z/determinant, x/determinant,
            (z*v-w*u)/determinant, (y*u-x*v)/determinant,
        )

    def operands(values):
        return [FloatObject(float(v)) for v in values]

    def glyph_unicode(name):
        name = str(name).lstrip("/").split(".", 1)[0]
        if not name or name == ".notdef":
            return ""
        if "_" in name:
            return "".join(glyph_unicode(part) for part in name.split("_"))
        mapped = adobe_glyphs.get("/" + name)
        if mapped is not None:
            return mapped
        if re.fullmatch(r"uni(?:[0-9A-Fa-f]{4})+", name):
            return bytes.fromhex(name[3:]).decode("utf-16-be")
        if re.fullmatch(r"u[0-9A-Fa-f]{4,6}", name):
            return chr(int(name[1:], 16))
        raise ValueError(f"Unknown Adobe glyph name: {name}")

    def unicode_stream(font):
        encoding = font.get("/Encoding")
        encoding = encoding.get_object() if hasattr(encoding, "get_object") else encoding
        base = str(encoding.get("/BaseEncoding", "/StandardEncoding")) if isinstance(encoding, dict) else str(encoding)
        if base != "/WinAnsiEncoding":
            raise ValueError(f"Unmapped native font uses unsupported encoding {base}.")
        mapping = {i: charset_encoding[base][i] for i in range(32, 256)}
        # PDF WinAnsi's glyph table differs from CP1252 at these positions.
        for code in [127, 129, 141, 143, 144, 157]:
            mapping[code] = "\u2022"
        mapping[160], mapping[173] = " ", "-"
        if isinstance(encoding, dict):
            code = None
            for item in encoding.get("/Differences", []):
                if isinstance(item, (int, float)):
                    code = int(item)
                elif code is not None and 0 <= code <= 255:
                    mapped = glyph_unicode(item)
                    if mapped:
                        mapping[code] = mapped
                    else:
                        mapping.pop(code, None)
                    code += 1
                else:
                    raise ValueError("Malformed font Differences array.")
        entries = [f"<{i:02X}> <{text.encode('utf-16-be').hex().upper()}>" for i, text in sorted(mapping.items())]
        blocks = [f"{len(entries[i:i+100])} beginbfchar\n" + "\n".join(entries[i:i+100]) + "\nendbfchar" for i in range(0, len(entries), 100)]
        text = (
            "/CIDInit /ProcSet findresource begin\n12 dict begin\nbegincmap\n"
            "/CIDSystemInfo << /Registry (Adobe) /Ordering (UCS) /Supplement 0 >> def\n"
            "/CMapName /NativeArialToUnicode def\n/CMapType 2 def\n"
            "1 begincodespacerange\n<00> <FF>\nendcodespacerange\n"
            + "\n".join(blocks)
            + "\nendcmap\nCMapName currentdict /CMap defineresource pop\nend\nend\n"
        )
        stream = DecodedStreamObject()
        stream.set_data(text.encode("ascii"))
        return stream

    original_signature = private_signature(reader)
    writer = PdfWriter(clone_from=reader)
    patched_fonts = 0
    clip_names = []
    # Replay state-only operations after Q: native PDFs reuse state across OCGs.
    state_ops = {
        b"q", b"Q", b"cm", b"w", b"J", b"j", b"M", b"d", b"ri", b"i", b"gs",
        b"CS", b"cs", b"SC", b"SCN", b"sc", b"scn", b"G", b"g", b"RG", b"rg", b"K", b"k",
        b"BT", b"ET", b"Tf", b"Tc", b"Tw", b"Tz", b"TL", b"Tr", b"Ts", b"Tm", b"Td", b"TD", b"T*",
    }
    identity = (1, 0, 0, 1, 0, 0)
    def is_true(value):
        return bool(getattr(value, "value", value))

    inlined_circles = 0
    removed_page_clips = 0
    for page in writer.pages:
        resources = page["/Resources"]
        if resources.get("/Pattern") or resources.get("/Shading"):
            raise ValueError("Repair expects the reviewed flat native vector PDF workflow.")
        for reference in resources.get("/Font", {}).values():
            font = reference.get_object()
            if "/ToUnicode" not in font:
                if str(font.get("/Subtype")) not in {"/TrueType", "/Type1"}:
                    raise ValueError("Missing Unicode map on a non-simple font.")
                font[NameObject("/ToUnicode")] = writer._add_object(unicode_stream(font).flate_encode())
                patched_fonts += 1
        content = ContentStream(page.get_contents(), writer)
        operations = content.operations
        if any(op in {b"W*", b"BI", b"INLINE IMAGE", b"sh"} for _, op in operations):
            raise ValueError("Unsupported clipping, raster, or shading in source.")

        def normal_alpha(gs):
            if str(gs.get("/BM", "/Normal")) != "/Normal" or str(gs.get("/SMask", "/None")) != "/None":
                raise ValueError("Only ordinary Normal alpha is reviewed for circle Forms.")
            if is_true(gs.get("/AIS", False)):
                raise ValueError("Alpha-is-shape Form compositing is not supported.")
            return float(gs.get("/ca", 1)), float(gs.get("/CA", 1))

        def flatten_circle(reference, caller_alpha):
            form = reference.get_object()
            if str(form.get("/Subtype")) != "/Form" or any(k in form for k in ["/SMask", "/Mask", "/ImageMask"]):
                raise ValueError("Only reviewed circle Form XObjects can be inlined.")
            group = form.get("/Group", {}).get_object() if hasattr(form.get("/Group", {}), "get_object") else form.get("/Group", {})
            if str(group.get("/S")) != "/Transparency" or is_true(group.get("/I", False)) or is_true(group.get("/K", False)):
                raise ValueError("Unsupported circle Form transparency group.")
            local = form.get("/Resources", {}).get_object()
            if set(local) - {"/ExtGState", "/ProcSet"}:
                raise ValueError("Circle Form must have no fonts, images, or nested resources.")
            body = ContentStream(form, writer).operations
            template = [b"rg", b"ri", b"gs", b"q", b"cm", b"m", b"c", b"c", b"c", b"c", b"f", b"Q"]
            if [o for _, o in body] != template:
                raise ValueError("Form is not the reviewed one-fill closed-circle template.")
            last = body[9][0][-2:]
            if any(abs(float(a)-float(b)) > 1e-8 for a, b in zip(last, body[5][0])):
                raise ValueError("Circle Form curve is not closed.")
            body_ctm = tuple(map(float, body[4][0]))
            points = [body[5][0]] + [args[j:j+2] for args, op in body if op == b"c" for j in [0, 2, 4]]
            mapped = [(body_ctm[0]*float(x)+body_ctm[2]*float(y)+body_ctm[4], body_ctm[1]*float(x)+body_ctm[3]*float(y)+body_ctm[5]) for x, y in points]
            x0, y0, x1, y1 = map(float, form["/BBox"])
            low_x, high_x, low_y, high_y = min(x0,x1), max(x0,x1), min(y0,y1), max(y0,y1)
            if any(not(low_x-0.001 <= x <= high_x+0.001 and low_y-0.001 <= y <= high_y+0.001) for x, y in mapped):
                raise ValueError("Circle Form BBox clips its geometry; inlining is unsafe.")
            local_name = body[2][0][0]
            local_gs = local["/ExtGState"][local_name].get_object()
            inner_alpha = normal_alpha(local_gs)
            merged = DictionaryObject(dict(local_gs))
            merged[NameObject("/ca")] = FloatObject(caller_alpha[0]*inner_alpha[0])
            merged[NameObject("/CA")] = FloatObject(caller_alpha[1]*inner_alpha[1])
            name = NameObject(f"/CircleAlpha{inlined_circles+1}")
            resources["/ExtGState"][name] = writer._add_object(merged)
            copied = [(a, o) if o != b"gs" else ([name], o) for a, o in body]
            matrix = tuple(map(float, form.get("/Matrix", identity)))
            return [([], b"q"), (operands(matrix), b"cm")] + copied + [([], b"Q")]

        # Keep only explicit viewport-independent page clips and single-fill Forms.
        flattened, alpha, alpha_stack, ctm, ctm_stack, path = [], (1, 1), [], identity, [], []
        xobjects = resources.get("/XObject", {})
        for i, (args, op) in enumerate(operations):
            if op == b"q":
                alpha_stack.append(alpha); ctm_stack.append(ctm)
            elif op == b"Q":
                if not alpha_stack:
                    raise ValueError("Unbalanced native graphics state.")
                alpha, ctm = alpha_stack.pop(), ctm_stack.pop()
            elif op == b"cm":
                ctm = multiply(ctm, tuple(map(float, args)))
            elif op == b"gs":
                gs = resources["/ExtGState"][args[0]].get_object()
                normal_alpha(gs)
                alpha = (float(gs.get("/ca", alpha[0])), float(gs.get("/CA", alpha[1])))
            if op in {b"m", b"l", b"c", b"v", b"y", b"h", b"re"}:
                path.append((args, op))
            if op == b"W":
                if i+1 >= len(operations) or operations[i+1][1] != b"n":
                    raise ValueError("Native clipping must end with n.")
                if len(path) != 1 or path[0][1] != b"re":
                    raise ValueError("Only a redundant simple page rectangle can be removed.")
                x, y, width, height = map(float, path[0][0])
                corners = [(x,y),(x+width,y),(x+width,y+height),(x,y+height)]
                world = [(ctm[0]*xx+ctm[2]*yy+ctm[4],ctm[1]*xx+ctm[3]*yy+ctm[5]) for xx,yy in corners]
                if any(abs(a[0]-b[0]) > 1e-6 and abs(a[1]-b[1]) > 1e-6 for a,b in zip(world,world[1:]+world[:1])):
                    raise ValueError("Rotated clipping rectangle is not an exact page clip.")
                bounds = [min(p[0] for p in world),min(p[1] for p in world),max(p[0] for p in world),max(p[1] for p in world)]
                if max(abs(a-b) for a,b in zip(bounds,map(float,page.mediabox))) > 0.001:
                    raise ValueError("Native viewport clipping cannot be removed.")
                removed_page_clips += 1
                continue
            if op == b"Do":
                flattened.extend(flatten_circle(xobjects[args[0]], alpha))
                inlined_circles += 1
                continue
            flattened.append((args, op))
            if op in {b"n", b"f", b"f*", b"S", b"s", b"B", b"B*", b"b", b"b*"}:
                path = []
        if alpha_stack:
            raise ValueError("Unbalanced native graphics state.")
        resources.pop(NameObject("/XObject"), None)
        page_group = page.get("/Group")
        if page_group:
            page_group = page_group.get_object()
            if str(page_group.get("/S")) != "/Transparency" or is_true(page_group.get("/I", False)) or is_true(page_group.get("/K", False)) or str(page_group.get("/CS", "/DeviceRGB")) != "/DeviceRGB":
                raise ValueError("Unsupported page transparency group.")
            page.pop(NameObject("/Group"))
        # Illustrator can close a group's q scope just after the next OCG tag.
        # Move Q across adjacent nonpainting tags, never across artwork/state.
        operations = []
        for item in flattened:
            if item[1] == b"Q":
                position = len(operations)
                while position and operations[position-1][1] in {b"BDC", b"BMC", b"EMC"}:
                    position -= 1
                operations.insert(position, item)
            else:
                operations.append(item)
        properties = resources.get("/Properties", {})
        marked, spans, text_start, text_pairs = [], [], None, {}
        for i, (args, op) in enumerate(operations):
            if op == b"BT":
                if text_start is not None:
                    raise ValueError("Nested text object in native source.")
                text_start = i
            elif op == b"ET":
                if text_start is None:
                    raise ValueError("Unmatched ET in native source.")
                text_pairs[text_start] = i
                text_start = None
            elif op in {b"BDC", b"BMC"}:
                name = None
                if op == b"BDC" and str(args[0]) == "/OC":
                    prop = properties[args[1]].get_object() if isinstance(args[1], str) else args[1].get_object()
                    name = str(prop["/Name"])
                marked.append((i, name, text_start))
            elif op == b"EMC":
                if not marked:
                    raise ValueError("Unmatched EMC in native source.")
                start, name, opened_text = marked.pop()
                if name is not None:
                    spans.append([start, i, name, opened_text, text_start])
        if marked or text_start is not None or len(spans) != 4 or {s[2] for s in spans} != expected:
            raise ValueError("Expected four complete, distinct native OCG sections.")
        # Text-only native OCGs can have BT/ET immediately outside their tags.
        for span in spans:
            start, end, name, opened_text, remaining_text = span
            if opened_text is not None:
                span[0] = opened_text
            if remaining_text is not None:
                span[1] = text_pairs[remaining_text]
        spans.sort(key=lambda s: s[0])
        if any(a[1] >= b[0] for a, b in zip(spans, spans[1:])):
            raise ValueError("OCG sections share text or have nested boundaries.")
        begins = {s[0]: s for s in spans}
        ends = {s[1]: s for s in spans}
        ctm, stack, repaired, current = identity, [], [], None
        for i, (args, op) in enumerate(operations):
            if i in begins:
                current = begins[i]
                left, bottom, right, top = map(float, page.mediabox)
                repaired += [([], b"q"), (operands(inverse(ctm)), b"cm"),
                             (operands([left, bottom, right-left, top-bottom]), b"re"),
                             ([], b"W"), ([], b"n"), (operands(ctm), b"cm")]
            repaired.append((args, op))
            if op == b"q":
                stack.append(ctm)
            elif op == b"Q":
                if not stack:
                    raise ValueError("Unbalanced source graphics state.")
                ctm = stack.pop()
            elif op == b"cm":
                ctm = multiply(ctm, tuple(map(float, args)))
            if i in ends:
                start, end, name = current[:3]
                block = operations[start:end+1]
                depth = 0
                for _, original_op in block:
                    depth += (original_op == b"q") - (original_op == b"Q")
                    if depth < 0:
                        raise ValueError("Native q/Q scope crosses an OCG boundary.")
                if depth:
                    raise ValueError("Native q/Q scope crosses an OCG boundary.")
                repaired.append(([], b"Q"))
                repaired.extend((a, o) for a, o in block if o in state_ops)
                clip_names.append(name)
                current = None
        if stack:
            raise ValueError("Unbalanced source graphics state.")
        content.operations = repaired
        content.get_data()  # Serialize edited operators before inherited compression.
        page[NameObject("/Contents")] = writer._add_object(content.flate_encode())
    destination.parent.mkdir(parents=True, exist_ok=True)
    writer.write(destination)
    repaired_reader = PdfReader(destination, strict=True)
    if private_signature(repaired_reader) != original_signature:
        raise AssertionError("Illustrator private data or catalog payload changed.")
    for old, new in zip(reader.pages, repaired_reader.pages):
        if " ".join(old.extract_text().split()) != " ".join(new.extract_text().split()):
            raise AssertionError("Public PDF text changed during repair.")
    return {"patched_fonts": patched_fonts, "clip_names": clip_names,
            "inlined_circles": inlined_circles, "removed_page_clips": removed_page_clips,
            "private_payload_sha256": original_signature, "private_payload_unchanged": True}


## 6. Save and preview

The only new figures are Stage-violin.png (600 DPI) and Stage-violin.pdf. Numerical inputs remain in the shared H5AD; no third result file is created.

In [ ]:
#6. =========== Save the final figure
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig = make_violin_layout(data)
report = export_violin(fig)
# Set the file or directory path.
display(Image(filename=str(ROOT), width=950))
